# Kapitel 2 - Ett ML projekt från början till slut

#### 1. I kapitlet beskrivs en checklista med sju steg. Beskriv de sju stegen översiktligt. I verkligheten, följs dessa steg i en rak progression eller arbetar man generellt sett mer iterativt?
Rama in problemet — affärsmål, problemtyp.
Skaffa datan — hämta in.
Utforska datan (EDA) — fördelningar, saknade värden, korrelationer, endast träningsdata
Förbereda datan — rensa, koda, skala.
Välj och träna modeller — prova flera grovt, sålla fram några kandidater
Finjustera — hyperparametrar, ensembler; testsetet används sist och en gång
Presentera och produktionssätt — kommunicera, driftsätt, övervaka

Progressionen är inte rak. Arbetet är iterativt: EDA skickar dig tillbaka för mer data, dåliga modellresultat leder till omgjorda features, produktionen avslöjar fel antaganden i steg 1. Listan är en checklista, inte ett flödesschema.

#### 2. Vad menas med att en modell produktionssätts?
Modellen tas i verklig drift och används av riktiga användare på riktig data, automatiskt, istället för att köras manuellt.

#### 3. Vad är scikit-learn? Designprinciper, estimators, predictors, transformers
Scikit-learn är Pythons standardbibliotek för klassisk ML. Modeller, förbehandling, korsvalidering, utvärderingsmått och pipelines. Byggt på NumPy/SciPy, har öppen källkod.
Designprinciper:
Konsekvens — samma gränssnitt överallt, byte av modell är en rad kod
Inspektion — hyperparametrar och inlärda värden är öppna attribut
Begränsad objekthierarki — vanliga arrayer och Python-typer, inga specialklasser
Komposition — byggblock kombineras fritt i pipelines
Rimliga standardvärden — fungerar direkt utan konfiguration

Estimator — lär sig från data, har .fit(). Bredaste kategorin.
Predictor — gör prediktioner, har .predict(). Alla modeller.
Transformer — omvandlar data, har .transform() och .fit_transform(). T.ex. StandardScaler, PCA.

#### 4. Vad är TensorFlow och Keras?
TensorFlow — Googles ramverk för djupinlärning. Sköter tunga beräkningar, automatisk derivering för backpropagation, GPU/TPU-körning, distribuerad träning och driftsättning. Kraftfullt men omständligt att programmera direkt.

Keras — högnivå-API ovanpå TensorFlow. Låter dig beskriva nätverk lager för lager i några rader kod, med .fit()-syntax lånad från scikit-learn. Ingår i TensorFlow som tf.keras.

#### Resonemangfrågor: 5. Kalle och Stina...
Stina har rätt. Testdatans värde ligger helt i att den är osedd. Så fort du justerar modellen utifrån testresultatet har den påverkat modellen.

#### 6. Många AI/ML projekt uppnår inte de ursprungligen satta målen eller att ens passera någon form av prototyp-stadie. Vad tror du detta beror på och hur ska vi förhålla oss till det?

Fel problem — projektet startas för att "göra något med AI", utan ett konkret mål.(Börja i problemet, inte tekniken.)
Datan håller inte — utspridd, ofullständig, för liten.(Undersök datan tidigt)

#### 8. Förklara vad koden nedan gör. Varför är det viktigt att kunna spara en modell?

In [1]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)

model = LinearRegression().fit(X, y)

dump(model, "linear_model.joblib")

model_loaded = load("linear_model.joblib")
print(model_loaded.predict(X[:5]))

[-93.58798172  16.81576383  38.11434    -55.22689649  24.16042231]


#### 8.
make_regression() skapar syntetisk data: 20 000 rader, 3 variabler, lite brus
LinearRegression().fit(X, y) skapar och tränar modellen i ett svep
dump(model, "linear_model.joblib") serialiserar modellen till fil — allt den lärt sig följer med
load(...) läser tillbaka den som ett fungerande modellobjekt, utan ny träning
predict(X[:5]) visar att den inlästa modellen fungerar
Varför spara modeller:

Träning är dyrt — timmar eller dagar för större modeller. Träna en gång, använd många.
Krävs för produktionssättning — ett API laddar modellen vid uppstart; utan sparande finns ingen väg från notebook till system
Reproducerbarhet — den sparade filen är exakt den modell som gav resultaten
Versionshantering — jämför versioner, rulla tillbaka vid behov
Delning — modellen kan skickas utan att träningsdatan följer med

#### 9....

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

df = pd.read_csv("../data/data_01.csv")
df.head()

,x1,x2,x3,x4,x5,target
0,0.743487,1.072825,1.332911,-1.244771,0.344978,220.173943
1,0.835264,0.202184,0.966480,0.745883,-0.033773,175.873929
2,-1.103234,0.030615,-0.140385,0.727683,-2.831224,-162.270054
3,1.210186,1.685258,-0.394123,0.719024,-2.166585,165.930461
4,0.474577,0.647737,-0.451812,-0.409472,-0.051473,43.250511


In [3]:
X = df.drop(columns="target")
y = df["target"]

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42)

print(len(X_train), len(X_val), len(X_test))

135 24 40


In [5]:
lin_reg = LinearRegression().fit(X_train, y_train)
tree_reg = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)

In [6]:
for model in [lin_reg, tree_reg]:
    rmse = root_mean_squared_error(y_val, model.predict(X_val))
    print(f"{type(model).__name__}: RMSE {rmse:.2f}")

LinearRegression: RMSE 3.59
DecisionTreeRegressor: RMSE 99.48


In [7]:
final_model = LinearRegression().fit(X_train_full, y_train_full)

In [8]:
rmse_test = root_mean_squared_error(y_test, final_model.predict(X_test))
print(f"Test RMSE: {rmse_test:.2f}")

Test RMSE: 3.37


In [9]:
production_model = LinearRegression().fit(X, y)

#### 10. 
Salary ska vara y. Erfarenhet kommer först och lönen följer av erfarenheten, inte tvärtom.

In [10]:
df = pd.read_csv("../data/salary_dataset.csv")
print(df.shape)
df.head()

(30, 2)


,YearsExperience,Salary
0,1.2,39344.0
1,1.4,46206.0
2,1.6,37732.0
3,2.1,43526.0
4,2.3,39892.0


In [11]:
X = df[["YearsExperience"]]
y = df["Salary"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(len(X_train), len(X_test))

24 6


In [13]:
from sklearn.model_selection import cross_validate

models = [LinearRegression(), DecisionTreeRegressor(random_state=42)]

for model in models:
    cv_results = cross_validate(
        model, X_train, y_train,
        scoring="neg_root_mean_squared_error",
        cv=5)
    
    rmse_scores = -cv_results["test_score"]
    print(f"{type(model).__name__}: "
          f"RMSE {rmse_scores.mean():.0f} (± {rmse_scores.std():.0f})")

LinearRegression: RMSE 5293 (± 2037)
DecisionTreeRegressor: RMSE 5612 (± 2119)


In [14]:
final_model = LinearRegression().fit(X_train, y_train)

rmse_test = root_mean_squared_error(y_test, final_model.predict(X_test))
print(f"Test RMSE: {rmse_test:.0f}")

print(f"Koefficient: {final_model.coef_[0]:.0f}")
print(f"Intercept: {final_model.intercept_:.0f}")

Test RMSE: 7059
Koefficient: 9424
Intercept: 24380


Linjär regression presterade något bättre i korsvalideringen (RMSE ~5 300 mot 
~5 600) och valdes därför. På testsetet blev RMSE ~7 059, alltså sämre än 
korsvalideringen antydde. Med endast 6 testobservationer är siffran dock osäker.

Koefficienten på ~9 424 innebär att modellen skattar värdet av ett års extra 
erfarenhet till ungefär 9 400 kr. Interceptet ~24 380 motsvarar skattad lön 
vid noll års erfarenhet.

#### 11. I denna uppgift kommer vi arbeta med kategorisk data.

In [15]:
import seaborn as sns

df = sns.load_dataset("mpg")
print(df.shape)
print(df.isna().sum())

(398, 9)
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64


In [16]:
df = df.dropna()
df = df.drop(columns="name")
df = pd.get_dummies(df, columns=["origin"], drop_first=True)
print(df.columns.tolist())

['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin_japan', 'origin_usa']


In [17]:
from sklearn.metrics import r2_score

X = df.drop(columns="mpg")
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"R²:   {r2_score(y_test, y_pred):.3f}")

RMSE: 3.26
R²:   0.792


In [18]:
for name, coef in zip(X.columns, model.coef_):
    print(f"{name:>15}: {coef:8.3f}")

      cylinders:   -0.342
   displacement:    0.019
     horsepower:   -0.022
         weight:   -0.006
   acceleration:    0.042
     model_year:    0.797
   origin_japan:    0.330
     origin_usa:   -2.875


#### 12. Förbättra huspris-modellen

In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

housing_original = pd.read_csv("../data/housing.csv")

housing = housing_original[housing_original["ocean_proximity"] != "ISLAND"]
housing = pd.get_dummies(housing, columns=["ocean_proximity"],
                         dtype=int, prefix="dmy")

train_full, test = train_test_split(housing, test_size=0.2, random_state=40)
train, val = train_test_split(train_full, test_size=0.25, random_state=36)

train = train.dropna()
val = val.dropna()
test = test.dropna()

X_train, y_train = train.drop(columns=["median_house_value"]), train["median_house_value"]
X_val, y_val = val.drop(columns=["median_house_value"]), val["median_house_value"]

print(X_train.shape, X_val.shape)

(12261, 12) (4084, 12)


In [20]:
grid_base = {
    "max_depth": [5, 10, 15, 50],
    "n_estimators": [1, 5, 10]
}

gs_base = GridSearchCV(RandomForestRegressor(random_state=42),
                       grid_base,
                       scoring="neg_root_mean_squared_error",
                       cv=5, n_jobs=-1)
gs_base.fit(X_train, y_train)

rmse_base = root_mean_squared_error(y_val, gs_base.predict(X_val))
print("Bästa parametrar:", gs_base.best_params_)
print(f"Baseline RMSE: {rmse_base:.0f}")

Bästa parametrar: {'max_depth': 50, 'n_estimators': 10}
Baseline RMSE: 52127


In [21]:
# Förbättring: bredare grid med fler träd och max_features
grid_ny = {
    "max_depth": [15, 30, None],
    "n_estimators": [100, 200],
    "max_features": [3, 5, 8]
}

gs_ny = GridSearchCV(RandomForestRegressor(random_state=42),
                     grid_ny,
                     scoring="neg_root_mean_squared_error",
                     cv=3, n_jobs=-1)
gs_ny.fit(X_train, y_train)

rmse_ny = root_mean_squared_error(y_val, gs_ny.predict(X_val))
print("Bästa parametrar:", gs_ny.best_params_)
print(f"Förbättrad RMSE: {rmse_ny:.0f}")

Bästa parametrar: {'max_depth': 30, 'max_features': 5, 'n_estimators': 200}
Förbättrad RMSE: 49376


In [22]:
print(f"Baseline:    {rmse_base:.0f}")
print(f"Förbättrad:  {rmse_ny:.0f}")
print(f"Förbättring: {rmse_base - rmse_ny:.0f} "
      f"({(rmse_base - rmse_ny) / rmse_base * 100:.1f}%)")

Baseline:    52127
Förbättrad:  49376
Förbättring: 2751 (5.3%)


Den bredare griden gav RMSE 49376 mot baseline 52127, en förbättring på cirka 5%. 
Bästa parametrar blev max_depth=30, max_features=5 och n_estimators=200. Den största vinsten kom från att tillåta fler träd.